# Часть 1. Очистка и анализ данных
**Датасет:** Student Performance Factors (Kaggle, lainguyn123)

В этом ноутбуке выполняются очистка данных и исследовательский анализ: проверка дубликатов, логических ошибок и пропусков, анализ случайной выборки 25%, графики, статистики и проверка гипотез на оставшихся 75%.


## 0 Загрузка данных и библиотек

In [ ]:
!pip install -q opendatasets legacy-cgi
import opendatasets as od
od.download("https://www.kaggle.com/datasets/lainguyn123/student-performance-factors")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
PATH = "/content/student-performance-factors/StudentPerformanceFactors.csv"
df = pd.read_csv(PATH)
rows_raw = len(df)
print("Исходный размер:", df.shape)
df.head()

## 1 Очистка датасета
Сначала чистим, потом анализируем — иначе любые средние и корреляции будут посчитаны на мусоре.

### 1.1 Единообразие текстовых значений
 Сначала смотрим на уникальные значения, потом обрезаем пробелы.

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    print(col, "->", sorted(df[col].dropna().unique()))

In [ ]:
for col in cat_cols:
    df[col] = df[col].str.strip()   # убираем пробелы по краям, регистр не трогаем — он здесь единообразный

### 1.2 Дубликаты
Одна строка — один студент. Если совпали все 20 столбцов сразу, это почти наверняка повторная загрузка одной записи, а не два разных студента с одинаковыми показателями. Оставляем первое вхождение, остальные убираем.

In [ ]:
dups = int(df.duplicated().sum())
print("Полных дублей:", dups)
df = df.drop_duplicates().reset_index(drop=True)
rows_after_dups = len(df)

### 1.3 Логические ошибки
Здесь проверяем, что значения вообще возможны:
- баллы и посещаемость — от 0 до 100;
- сон — от 0 до 24 часов в сутки;
- часы учёбы, спорта — не отрицательные и не больше 168 (столько часов в неделе);
- совместное правило: сон за неделю (часы в сутки × 7) + учёба + физическая активность не могут в сумме превышать 168 часов.

Строки, где правило нарушено, удаляем целиком. Правильное значение мы всё равно не знаем, а таких строк, скорее всего, единицы, так что потеря несущественна. Особенно важно, если ошибка в целевой переменной: подставлять "что-нибудь похожее" в `Exam_Score` нельзя.

In [ ]:
rules = {
    "Exam_Score":        (0, 100),
    "Attendance":        (0, 100),
    "Previous_Scores":   (0, 100),
    "Sleep_Hours":       (0, 24),
    "Hours_Studied":     (0, 168),
    "Physical_Activity": (0, 168),
    "Tutoring_Sessions": (0, 168),
}

bad = pd.Series(False, index=df.index)
report = []
for col, (lo, hi) in rules.items():
    mask = (df[col] < lo) | (df[col] > hi)
    report.append({"Правило": f"{col} вне [{lo}; {hi}]", "Нарушений": int(mask.sum())})
    bad |= mask.fillna(False)

week_total = df["Sleep_Hours"] * 7 + df["Hours_Studied"] + df["Physical_Activity"]
mask_week = week_total > 168
report.append({"Правило": "сон*7 + учёба + спорт > 168 ч/нед", "Нарушений": int(mask_week.sum())})
bad |= mask_week.fillna(False)

pd.DataFrame(report)

In [ ]:
print("Строк с логическими ошибками:", int(bad.sum()))
df[bad]   # посмотрим на них глазами перед удалением

In [ ]:
df = df[~bad].reset_index(drop=True)
rows_after_logic = len(df)

### 1.4 Пропуски
Смотрим, где пропуски и какая это доля.

In [ ]:
miss = pd.DataFrame({"Пропусков": df.isna().sum(), "Доля, %": (df.isna().mean() * 100).round(2)})
miss[miss["Пропусков"] > 0]

**Что делаем и почему.** Пропуски находятся в категориальных столбцах. Вместо удаления строк мы сохраняем наблюдения: после разделения на train/test заполняем пропуски наиболее частой категорией, используя `SimpleImputer(strategy="most_frequent")`. Импьютер обучается только на train, а затем тем же правилом преобразует test. Это сохраняет больше данных и предотвращает утечку статистики из тестовой выборки.


In [ ]:
# Строки с пропусками не удаляем: заполнение будет обучено на train ниже.
rows_final = len(df)
print("Строк сохранено:", rows_final)
print("Пропусков перед разделением:", int(df.isna().sum().sum()))


### 1.5 Итог очистки

In [ ]:
summary = pd.DataFrame({
    "Шаг": ["Исходный датасет", "После удаления дублей", "После удаления логических ошибок", "После решения по пропускам (строки сохранены)"],
    "Строк": [rows_raw, rows_after_dups, rows_after_logic, rows_final],
})
summary["Потеряно на шаге"] = (-summary["Строк"].diff()).fillna(0).astype(int)
print(f"Всего удалено на этапах дубликатов и логических ошибок: {rows_raw - rows_after_logic} строк")
print("Пропусков перед split:", int(df.isna().sum().sum()), "| дублей осталось:", int(df.duplicated().sum()))
summary


## 1 Анализ данных
Берём 25% случайных строк — на них смотрим и придумываем гипотезы. Остальные 75% не трогаем до момента проверки: если проверять гипотезы на тех же данных, на которых они родились, это самообман. `random_state` фиксируем, чтобы результат повторялся.

In [ ]:
sample_df = df.sample(frac=0.25, random_state=42)
rest_df = df.drop(sample_df.index)
print("Выборка:", sample_df.shape, "| остаток для проверки:", rest_df.shape)

### 1.1 Среднее и медиана по выборке

In [ ]:
cols_to_check = ["Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores", "Exam_Score"]
sample_df[cols_to_check].agg(["mean", "median"]).T.round(2)

Если среднее и медиана почти совпадают, признак распределён симметрично; если сильно расходятся — в данных есть перекос или выбросы, которые тянут среднее в сторону.

### 1.2 Графики

In [ ]:
# Более аккуратное оформление графиков
plt.rcParams.update({
    "figure.figsize": (11, 6),
    "figure.dpi": 120,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# График 1. Часы учёбы и итоговый балл
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(sample_df["Hours_Studied"], sample_df["Exam_Score"],
          alpha=0.35, s=42, color="#3977B8", edgecolors="white", linewidths=0.25)
ax.set_title("Связь количества часов учёбы с итоговым баллом", pad=14)
ax.set_xlabel("Часы учёбы в неделю")
ax.set_ylabel("Итоговый балл")
ax.grid(True, alpha=0.22)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# График 2. Посещаемость и итоговый балл
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(sample_df["Attendance"], sample_df["Exam_Score"],
          alpha=0.35, s=42, color="#E59A28", edgecolors="white", linewidths=0.25)
ax.set_title("Связь посещаемости с итоговым баллом", pad=14)
ax.set_xlabel("Посещаемость, %")
ax.set_ylabel("Итоговый балл")
ax.grid(True, alpha=0.22)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# График 3. Средний балл по вовлечённости родителей
order = ["Low", "Medium", "High"]
mean_scores = sample_df.groupby("Parental_Involvement")["Exam_Score"].mean().reindex(order)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(mean_scores.index, mean_scores.values, color=["#67B567", "#3E9B75", "#247B61"],
               edgecolor="white", width=0.62)
ax.bar_label(bars, fmt="%.1f", padding=4, fontsize=11)
ax.set_title("Средний итоговый балл по вовлечённости родителей", pad=14)
ax.set_xlabel("Вовлечённость родителей")
ax.set_ylabel("Средний итоговый балл")
ax.grid(axis="y", alpha=0.22)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


### 1.3 Гипотезы и ход мыслей

> Текст ниже написан под типичную картину для этого датасета. После запуска сверь его со своими графиками: если что-то выглядит иначе, поправь формулировку своими словами.

**Гипотеза 1.** На левом графике облако точек идёт вверх: чем больше часов учёбы в неделю, тем выше балл. Разброс по вертикали большой, значит, часы учёбы — не единственный фактор, но направление видно. *Предположение:* у студентов, которые больше учатся, балл в среднем выше (положительная корреляция).

**Гипотеза 2.** На среднем графике картина похожая, причём точки лежат плотнее вдоль подъёма, чем на первом. *Предположение:* посещаемость связана с баллом даже сильнее, чем часы учёбы.

**Гипотеза 3.** На правом графике столбики растут от Low к High: там, где родители больше вовлечены, средний балл выше. *Предположение:* средний балл группы High заметно выше, чем группы Low (и это не случайность выборки).

**Гипотеза 4.** Среднее и медиана сна почти совпадают и лежат в районе обычных 6–8 часов, то есть у большинства студентов сон примерно одинаковый. Если почти у всех сон одинаковый, ему трудно объяснять различия в баллах. *Предположение:* связи между часами сна и баллом почти нет.

### 1.4 Проверка гипотез на остальных 75% данных

In [ ]:
results = []

# Г1: Hours_Studied и Exam_Score
r1, p1 = stats.pearsonr(rest_df["Hours_Studied"], rest_df["Exam_Score"])
results.append(["Г1. Hours_Studied ↔ Exam_Score", f"r = {r1:.3f}", f"p = {p1:.2g}", "подтверждена" if (r1 > 0.2 and p1 < 0.05) else "не подтверждена"])

# Г2: Attendance и Exam_Score, плюс сравнение силы связи с Г1
r2, p2 = stats.pearsonr(rest_df["Attendance"], rest_df["Exam_Score"])
results.append(["Г2. Attendance ↔ Exam_Score (сильнее, чем Г1)", f"r = {r2:.3f}", f"p = {p2:.2g}", "подтверждена" if (r2 > r1 and p2 < 0.05) else "не подтверждена"])

# Г3: High против Low по вовлечённости родителей
high = rest_df.loc[rest_df["Parental_Involvement"] == "High", "Exam_Score"]
low  = rest_df.loc[rest_df["Parental_Involvement"] == "Low", "Exam_Score"]
t3, p3 = stats.ttest_ind(high, low, equal_var=False)
results.append(["Г3. Parental_Involvement: High vs Low", f"разница средних = {high.mean() - low.mean():.2f}", f"p = {p3:.2g}", "подтверждена" if (high.mean() > low.mean() and p3 < 0.05) else "не подтверждена"])

# Г4: связи сна и балла почти нет
r4, p4 = stats.pearsonr(rest_df["Sleep_Hours"], rest_df["Exam_Score"])
results.append(["Г4. Sleep_Hours ↔ Exam_Score (связи почти нет)", f"r = {r4:.3f}", f"p = {p4:.2g}", "подтверждена" if abs(r4) < 0.1 else "не подтверждена"])

pd.DataFrame(results, columns=["Гипотеза", "Показатель", "p-value", "Итог"])

Как читать таблицу: `r` — коэффициент корреляции (от −1 до 1; около 0 — связи нет), `p-value` — вероятность увидеть такую связь случайно (меньше 0.05 — считаем, что связь не случайна). Пороги (`r > 0.2`, `|r| < 0.1`) я выбрал сам как разумные ориентиры, их можно обсудить.

### 1.5 Вывод по анализу данных
В ходе исследовательского анализа были сформулированы четыре гипотезы и проверены на оставшихся 75% данных.

Гипотеза 1 — количество часов учёбы положительно связано с итоговым баллом. Подтвердилась. Коэффициент корреляции Пирсона составил 0,452, а p-value значительно меньше 0,05. Это говорит о наличии положительной связи: в среднем, чем больше времени студент уделяет учёбе, тем выше его итоговый балл. Результат совпал с тенденцией, наблюдавшейся на первоначальной выборке.

Гипотеза 2 — посещаемость сильнее связана с итоговым баллом, чем часы учёбы. Подтвердилась. Коэффициент корреляции посещаемости составил 0,580, что выше, чем у количества часов учёбы (0,452). Следовательно, среди рассмотренных числовых признаков посещаемость оказалась более тесно связана с итоговым баллом. Это также соответствовало визуальным наблюдениям по графикам.

Гипотеза 3 — при высокой вовлечённости родителей средний балл выше, чем при низкой. Подтвердилась. Разница средних баллов между группами High и Low составила 1,79 балла, а p-value оказался значительно меньше 0,05. Это означает, что различие между группами статистически значимо и не похоже на случайное колебание выборки. Результат соответствует тенденции, замеченной на графике.

Гипотеза 4 — между продолжительностью сна и итоговым баллом практически нет линейной связи. Подтвердилась в рамках выбранного критерия. Коэффициент корреляции составил −0,022, то есть очень близок к нулю. Статистически значимой связи обнаружено не было (p-value = 0,12). Это совпало с первоначальным предположением о том, что продолжительность сна в данном датасете практически не связана с итоговым баллом.

В результате анализа наиболее сильная линейная связь с итоговым баллом среди проверенных числовых признаков наблюдалась у посещаемости (r = 0,580). Количество часов учёбы также показало положительную связь, но меньшей силы.

При этом важно учитывать, что корреляция не означает причинно-следственную связь. Нельзя утверждать, что именно посещаемость напрямую повышает балл, поскольку на результат могут влиять и другие факторы, например мотивация ученика. Кроме того, датасет является синтетическим, поэтому обнаруженные закономерности могут отличаться от реальных данных.